# WikiGPT 02: Pretrain

Trains the 12.6M parameter model on the Wikipedia tokens from notebook 01. Writes `ckpt.pt`, `log.csv`, `loss.png`, `samples.txt` and `tokenizer.json` to `/kaggle/working/runs/pretrain`.

**Settings:** Accelerator GPU T4 x2, Internet On.  
**Inputs:** the `wikigpt-code` dataset and the output of notebook 01.  
**Run with:** Save Version, then Save & Run All (Commit).

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")


def find_input(pattern: str) -> Path:
    matches = sorted(INPUT.rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"{pattern} not found under {INPUT}. Attach the inputs listed at the top of this notebook.")
    return matches[0]


def run(*args) -> None:
    command = [sys.executable, "-u", *map(str, args)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"command failed with exit code {process.returncode}")


CODE = find_input("wikigpt/__init__.py").parents[1]
sys.path.insert(0, str(CODE))
sys.dont_write_bytecode = True
os.environ.update(PYTHONPATH=str(CODE), PYTHONDONTWRITEBYTECODE="1", HF_HUB_DISABLE_PROGRESS_BARS="1")
DATA = find_input("train.bin").parent

print("code:", CODE)
print("data:", DATA)
print("inputs:", ", ".join(sorted(p.name for p in INPUT.iterdir())))
print("torch:", torch.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print(f"free disk: {shutil.disk_usage(WORKING).free / 1e9:.1f} GB")

## Tests

In [ ]:
run("-m", "pytest", CODE / "tests", "-q", "-p", "no:cacheprovider")

## Smoke run

50 steps to check that the loss falls and to measure speed. Full run time is roughly 460M tokens divided by the tok/s shown here.

In [ ]:
SMOKE = WORKING / "smoke"
run(
    CODE / "scripts" / "pretrain.py",
    "--data", DATA,
    "--out", SMOKE,
    "--max-steps", 50,
    "--warmup-steps", 10,
    "--log-interval", 5,
    "--eval-interval", 25,
    "--eval-batches", 5,
    "--ckpt-interval", 50,
)
shutil.rmtree(SMOKE)

## Full run

To continue after a session ended: attach this notebook's previous output as an input and set `RESUME = True`.

In [ ]:
OUT = WORKING / "runs" / "pretrain"
RESUME = False

args = [CODE / "scripts" / "pretrain.py", "--data", DATA, "--out", OUT]
if RESUME:
    args += ["--resume", find_input("pretrain/ckpt.pt")]
run(*args)

## Results

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(OUT / "loss.png")))
print((OUT / "samples.txt").read_text(encoding="utf-8")[-4000:])